In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl transformers peft datasets pillow huggingface_hub \
             pyyaml bitsandbytes accelerate scikit-learn -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import whoami
print("Login sebagai:", whoami()["name"])

Login sebagai: Lifunn


In [4]:
!git clone https://github.com/Lifunn/Pipeline-Training-SFT-DFK.git /content/pipeline
%cd /content/pipeline

fatal: destination path '/content/pipeline' already exists and is not an empty directory.
/content/pipeline


In [5]:
import yaml

DRIVE = "/content/drive/MyDrive"

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["train_file"] = f"{DRIVE}/train_final_fixed.jsonl"
cfg["data"]["eval_file"]  = f"{DRIVE}/valid_final_fixed.jsonl"
cfg["data"]["image_dir"]  = f"{DRIVE}/hf_images_folder"

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)

print("Config updated ✓")
print(f"  train : {cfg['data']['train_file']}")
print(f"  eval  : {cfg['data']['eval_file']}")
print(f"  images: {cfg['data']['image_dir']}")

Config updated ✓
  train : /content/drive/MyDrive/train_final_fixed.jsonl
  eval  : /content/drive/MyDrive/valid_final_fixed.jsonl
  images: /content/drive/MyDrive/hf_images_folder


In [13]:
# Pull kode terbaru
!git pull origin main

# Force reload semua module pipeline
import importlib
import src.data_utils, src.model_utils, src.training_utils
importlib.reload(src.data_utils)
importlib.reload(src.model_utils)
importlib.reload(src.training_utils)

print("Modules reloaded ✓")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.79 KiB | 1.79 MiB/s, done.
From https://github.com/Lifunn/Pipeline-Training-SFT-DFK
 * branch            main       -> FETCH_HEAD
   d970537..9d1f140  main       -> origin/main
Updating d970537..9d1f140
Fast-forward
 src/data_utils.py | 70 ++++++++++++++++++++++++++++++++++++++-----------------
 1 file changed, 48 insertions(+), 22 deletions(-)
Modules reloaded ✓


In [14]:
import unsloth
!python train.py --smoke_test --hf_token $HF_TOKEN

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
19:07:54 [INFO] train — ─── Pre-flight ──────────────────────────────────────────────
19:07:54 [INFO] train —   GPU    : NVIDIA A100-SXM4-80GB (85.1 GB)
19:07:54 [INFO] httpx — HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
19:07:54 [WARNING] huggingface_hub._login — Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
19:07:54 [INFO] train —   HF     : Tok

In [20]:
import shutil

src = "/content/pipeline"
dst = "/content/drive/MyDrive/dfk_sft_output"

shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Model tersimpan ke Drive: {dst} ✓")

Model tersimpan ke Drive: /content/drive/MyDrive/dfk_sft_output ✓


# Full Training

In [ ]:
# !python train.py --hf_token $HF_TOKEN

# Evaluasi

In [ ]:
# !python evaluate.py \
#     --model_dir ./output/dfk_sft \
#     --max_samples 300 \
#     --output_json ./output/dfk_sft/eval_results.json \
#     --hf_token $HF_TOKEN